# Knowledge Graphs & GNNs — Finnish Building Context
### Adapted from `KG_GNN_AEC.ipynb` (ArchiColab)

---

## About this notebook

This notebook adapts the Swiss Dwellings KG+GNN pipeline to **Finnish open geodata**. Because the available Finnish open datasets operate at **building scale** (not room scale), the pipeline is deliberately re-framed:

| Original (Swiss Dwellings) | This notebook (Finnish) |
|---------------------------|-------------------------|
| Data unit: **room** (interior) | Data unit: **building** (exterior footprint) |
| Hierarchy: Site → Building → Floor → Apartment → Room | Hierarchy: Municipality → Residential Area → Building |
| KG edge to predict: `bot:adjacentZone` (room–room) | KG edge to predict: `bot:adjacentTo` (building–building) |
| Data source: Swiss Dwellings CSV | Data source: SYKE WFS + OpenStreetMap |

### Data sources used

| Source | What it provides | Access |
|--------|-----------------|--------|
| **SYKE / LIITERI** | Residential area polygons (kerrostaloalue, pientaloalue) | Free WFS, no key |
| **OpenStreetMap** (via `osmnx`) | Building footprint polygons + type tags | Free, no key |
| **NLS Finland** (optional) | Topographic buildings, LoD2 3D | Free OGC API, needs API key |

> **Honest data gap note:** Neither SYKE nor NLS Finland provides **room-level** interior data. For room-level analysis (exact replication of the original notebook), you need **IFC files**. See the Appendix for how to extend this pipeline to room-level using `ifcopenshell`.

---

## Pipeline overview

```
SYKE WFS           OpenStreetMap (osmnx)
  │                      │
  └──────── join ─────────┘
            │
            ▼
   Finnish Building KG (rdflib)
   Municipality → ResidentialArea → Building
            │
            ▼
   Compute spatial adjacency (shapely)
            │
            ▼
   HeteroData graph (PyTorch Geometric)
            │
            ▼
   GraphSAGE — predict building adjacency
            │
            ▼
   Evaluate (AUC-ROC) + Visualize
```

---
## Part 0 — Setup

### 📌 Instruction: Dependencies

New libraries compared to the original notebook:

| Library | Purpose | Install |
|---------|---------|--------|
| `geopandas` | Spatial dataframes (GeoDataFrame) | `pip install geopandas` |
| `osmnx` | Download OSM building footprints | `pip install osmnx` |
| `owslib` | WFS client (SYKE data) | `pip install owslib` |
| `folium` | Interactive map visualization | `pip install folium` |
| `pyproj` | Coordinate reprojection | `pip install pyproj` |
| `shapely` | Geometric operations (adjacency) | already in original notebook |

All other libraries (rdflib, torch-geometric, networkx) are the same as the original.

In [ ]:
# Install new dependencies (run once)
# !pip install geopandas osmnx owslib folium pyproj

In [ ]:
# ── Core spatial / data libraries ─────────────────────────────────────────────
import geopandas as gpd
import osmnx as ox
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
import folium
from pyproj import Transformer
import requests

# ── Knowledge Graph ────────────────────────────────────────────────────────────
import rdflib
from rdflib import Graph, Literal, RDF, RDFS, URIRef, Namespace

# ── Graph / GNN ────────────────────────────────────────────────────────────────
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from itertools import combinations
from copy import copy
import tqdm
import pickle, os, uuid, warnings
warnings.filterwarnings('ignore')

os.makedirs("data", exist_ok=True)
print(f"PyTorch {torch.__version__} | GeoPandas {gpd.__version__}")

### 📌 Instruction: Study area selection

We use **Helsinki** as the study area. You can change `CITY` to any Finnish city covered by OSM (Tampere, Turku, Espoo, Oulu, etc.).

`ADJACENCY_BUFFER_M`: two buildings are considered **adjacent** (sharing an urban block) if their footprints are within this distance of each other. This replaces the `IfcRelSpaceBoundary` adjacency from IFC files.

In [ ]:
# ── Study area parameters ──────────────────────────────────────────────────────
CITY              = "Helsinki, Finland"
CITY_SHORT        = "helsinki"          # used for filenames
DISTRICT          = "Kallio"            # sub-area for manageable dataset size
                                        # change to None to use entire city (slower)
ADJACENCY_BUFFER_M = 10                 # metres: buildings within this distance = adjacent
CRS_METRIC        = "EPSG:3067"         # Finnish national ETRS-TM35FIN (metres)
CRS_WGS84         = "EPSG:4326"         # WGS84 (lat/lon, used by OSM)

# SYKE WFS endpoint
SYKE_WFS_URL   = "https://paikkatiedot.ymparisto.fi/geoserver/liiteri_asuinalueet/wfs"
SYKE_LAYER_APT = "liiteri_asuinalueet:kerrostaloalue24"   # apartment block areas 2024
SYKE_LAYER_DTH = "liiteri_asuinalueet:pientaloalue24"     # detached-house areas 2024

print(f"Study area : {CITY}")
print(f"District   : {DISTRICT}")
print(f"Adjacency  : buildings within {ADJACENCY_BUFFER_M} m")

---
## Part 1 — Data Acquisition

### 1a — SYKE Residential Areas (WFS)

### 📌 Instruction: What SYKE provides

The **SYKE LIITERI** dataset classifies Finnish land into residential area types:

| Layer | Type | Finnish | Meaning |
|-------|------|---------|--------|
| `kerrostaloalue24` | `APT` | Kerrostaloalue | Apartment block area (≥4-storey) |
| `pientaloalue24` | `DTH` | Pientaloalue | Detached/semi-detached house area |
| `harvapientaloasutus24` | `SPR` | Harva pientaloasutus | Sparse rural housing |

These are **polygons** in EPSG:3067 (Finnish national CRS). We use them as the `fi:ResidentialArea` node type in the KG.

The WFS is publicly accessible — no API key required.

In [ ]:
def fetch_syke_layer(wfs_url: str, layer_name: str, bbox=None) -> gpd.GeoDataFrame:
    """
    Download a SYKE WFS layer as a GeoDataFrame.
    bbox: (minx, miny, maxx, maxy) in EPSG:3067 to limit the download area.
    """
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeName": layer_name,
        "outputFormat": "application/json",
        "srsName": "EPSG:3067",
        "count": 5000,   # limit rows
    }
    if bbox:
        params["bbox"] = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]},EPSG:3067"

    resp = requests.get(wfs_url, params=params, timeout=60)
    resp.raise_for_status()
    gdf = gpd.read_file(resp.text)
    gdf = gdf.set_crs(CRS_METRIC, allow_override=True)
    return gdf


# Helsinki bounding box in EPSG:3067 (approximate)
HELSINKI_BBOX = (381_000, 6_664_000, 406_000, 6_688_000)

print("Downloading SYKE apartment block areas (kerrostaloalue24)...")
gdf_apt = fetch_syke_layer(SYKE_WFS_URL, SYKE_LAYER_APT, bbox=HELSINKI_BBOX)
gdf_apt["area_type"] = "APT"   # apartment block

print("Downloading SYKE detached-house areas (pientaloalue24)...")
gdf_dth = fetch_syke_layer(SYKE_WFS_URL, SYKE_LAYER_DTH, bbox=HELSINKI_BBOX)
gdf_dth["area_type"] = "DTH"   # detached-house

# Combine both types
gdf_areas = gpd.GeoDataFrame(pd.concat([gdf_apt, gdf_dth], ignore_index=True),
                              crs=CRS_METRIC)

print(f"\nTotal residential areas: {len(gdf_areas)}")
print(gdf_areas[["area_type", "geometry"]].head())

In [ ]:
# Add a stable URI identifier to each area
gdf_areas["uri"] = [
    f"https://liiteri.ymparisto.fi/area/{area_type}_{i:05d}"
    for i, area_type in enumerate(gdf_areas["area_type"])
]

# Basic statistics
gdf_areas["area_m2"] = gdf_areas.geometry.area
print("Area statistics (m²):")
print(gdf_areas.groupby("area_type")["area_m2"].describe().round(0))

### 1b — Building Footprints via OpenStreetMap (osmnx)

### 📌 Instruction: Why OSM instead of NLS Finland?

**NLS Finland Topographic Database** has official building footprints but requires a free API key (register at maanmittauslaitos.fi). An API key approach is shown in the Appendix.

**OpenStreetMap (OSM)** via `osmnx` is:
- Free with no registration
- Has building type tags (`building=apartments`, `building=house`, etc.)
- Good coverage for Finnish cities

OSM buildings are the analog of **rooms** in the Swiss Dwellings notebook — they are the primary nodes in our KG.

`osmnx.features_from_place()` downloads all OSM features matching a tag filter within the named place boundary.

In [ ]:
# Download building footprints from OpenStreetMap
# Use DISTRICT for a manageable dataset size; change to CITY for full coverage
query_area = f"{DISTRICT}, {CITY}" if DISTRICT else CITY

print(f"Downloading buildings from OSM for: {query_area}")
buildings_raw = ox.features_from_place(
    query_area,
    tags={"building": True}   # all features tagged as a building
)

# Keep only polygon features (drop point entries)
buildings_raw = buildings_raw[
    buildings_raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

# Project to Finnish metric CRS for area and distance calculations
gdf_buildings = buildings_raw.to_crs(CRS_METRIC).reset_index(drop=True)

print(f"Buildings downloaded: {len(gdf_buildings)}")
print("Building type breakdown:")
print(gdf_buildings["building"].value_counts().head(10))

In [ ]:
# Clean and enrich building attributes
gdf_buildings = gdf_buildings.reset_index(drop=True)

# Assign stable URIs using OSM ID or a generated UUID
gdf_buildings["uri"] = [
    f"https://osm.org/building/{i}" for i in range(len(gdf_buildings))
]

# Normalise building type: map OSM tags to three simplified classes
APT_TAGS  = {"apartments", "residential", "yes", "flats", "dormitory"}
COMM_TAGS = {"commercial", "retail", "office", "hotel", "industrial"}
building_type_map = {
    t: ("APT" if t in APT_TAGS else "COMM" if t in COMM_TAGS else "OTHER")
    for t in gdf_buildings["building"].dropna().unique()
}
gdf_buildings["building_class"] = (
    gdf_buildings["building"].map(building_type_map).fillna("OTHER")
)

# Geometric features (will be node features for the GNN)
gdf_buildings["area_m2"]      = gdf_buildings.geometry.area.round(2)
gdf_buildings["perimeter_m"]  = gdf_buildings.geometry.length.round(2)
gdf_buildings["compactness"]  = (
    4 * np.pi * gdf_buildings["area_m2"] / (gdf_buildings["perimeter_m"] ** 2)
).round(4)  # 1.0 = circle; lower = more elongated

# Bounding box aspect ratio
bounds = gdf_buildings.geometry.bounds
width  = bounds["maxx"] - bounds["minx"]
height = bounds["maxy"] - bounds["miny"]
gdf_buildings["aspect_ratio"] = (width / height.replace(0, np.nan)).fillna(1).round(3)

# Centroid coordinates (normalised later for GNN)
gdf_buildings["centroid_x"] = gdf_buildings.geometry.centroid.x.round(1)
gdf_buildings["centroid_y"] = gdf_buildings.geometry.centroid.y.round(1)

print(f"Building dataset: {len(gdf_buildings)} buildings")
print(gdf_buildings[["building_class","area_m2","perimeter_m","compactness","aspect_ratio"]].describe().round(3))

### 1c — Spatial Join: Assign Buildings to Residential Areas

### 📌 Instruction: Joining the two datasets

The SYKE polygons define **neighbourhood context** (is a building in a kerrostaloalue or pientaloalue area?). We assign each building to the residential area whose polygon contains its centroid.

This creates the KG hierarchy: `ResidentialArea` → `hasPart` → `Building`

In [ ]:
# Spatial join: buildings to residential areas
# Use centroid to avoid partial-overlap issues with building footprints
building_centroids = gdf_buildings.copy()
building_centroids["geometry"] = gdf_buildings.geometry.centroid

joined = gpd.sjoin(
    building_centroids[["uri", "geometry"]],
    gdf_areas[["uri", "area_type", "geometry"]],
    how="left",
    predicate="within"
)

# Add area info back to buildings
gdf_buildings["area_uri"]  = joined["uri_right"].values
gdf_buildings["area_type"] = joined["area_type"].values

# Buildings not in any SYKE area get a generic label
gdf_buildings["area_uri"]  = gdf_buildings["area_uri"].fillna("https://liiteri.ymparisto.fi/area/UNKNOWN")
gdf_buildings["area_type"] = gdf_buildings["area_type"].fillna("UNKNOWN")

print(f"Buildings assigned to a SYKE area: {(gdf_buildings['area_type'] != 'UNKNOWN').sum()}")
print(f"Buildings without area match      : {(gdf_buildings['area_type'] == 'UNKNOWN').sum()}")
print("\nBuildings per area type:")
print(gdf_buildings["area_type"].value_counts())

### 1d — Compute Building Adjacency

### 📌 Instruction: Defining adjacency without IFC

In the original notebook, adjacency came from `bot:adjacentZone` in IFC files (`IfcRelSpaceBoundary`). Here, we compute it **geometrically**:

> Two buildings are **adjacent** if their footprints are within `ADJACENCY_BUFFER_M` metres of each other AND they belong to the same residential area.

This is the ground truth for the link prediction task. In a real scenario, this adjacency information could be missing (e.g., you have footprints from a PDF site plan but no topology). The GNN then learns to reconstruct it from geometric features alone.

We use `shapely.buffer()` — expand each polygon by the threshold, then check for intersections.

In [ ]:
def compute_adjacency(gdf: gpd.GeoDataFrame, buffer_m: float) -> list[tuple]:
    """
    Return list of (idx_i, idx_j) pairs where buildings are spatially adjacent.
    Two buildings are adjacent if their buffered footprints intersect.
    Only pairs within the same area_uri are checked (same neighbourhood).
    """
    adjacent_pairs = []
    buffered = gdf.geometry.buffer(buffer_m)

    # Group by residential area to limit the search space
    for area_uri, group in gdf.groupby("area_uri"):
        idxs = list(group.index)
        for i, j in combinations(idxs, 2):
            if buffered[i].intersects(buffered[j]):
                adjacent_pairs.append((i, j))

    return adjacent_pairs


print(f"Computing building adjacency (buffer = {ADJACENCY_BUFFER_M} m)...")
print("This may take 1-2 minutes for large datasets...")

# Only process buildings that were successfully assigned to a SYKE area
gdf_in_area = gdf_buildings[gdf_buildings["area_type"] != "UNKNOWN"].copy().reset_index(drop=True)
adjacent_pairs = compute_adjacency(gdf_in_area, ADJACENCY_BUFFER_M)

print(f"\nBuildings in SYKE areas  : {len(gdf_in_area)}")
print(f"Adjacent building pairs  : {len(adjacent_pairs)}")
print(f"Avg adjacency per building: {2*len(adjacent_pairs)/len(gdf_in_area):.1f}")

### 1e — Visualize the Study Area

### 📌 Instruction: Map overview

This map shows:
- **SYKE residential area polygons** (coloured by type: blue=kerrostaloalue, orange=pientaloalue)
- **OSM building footprints** (grey)
- **Adjacency edges** (red lines between adjacent building centroids)

In [ ]:
# Reproject to WGS84 for folium
areas_wgs = gdf_areas.to_crs(CRS_WGS84)
bldg_wgs  = gdf_in_area.to_crs(CRS_WGS84)

# Centre of map
centre = [bldg_wgs.geometry.centroid.y.mean(), bldg_wgs.geometry.centroid.x.mean()]
m = folium.Map(location=centre, zoom_start=14, tiles="CartoDB positron")

# SYKE areas
color_map = {"APT": "#4472C4", "DTH": "#ED7D31", "UNKNOWN": "#999"}
for _, row in areas_wgs.iterrows():
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        style_function=lambda f, at=row["area_type"]: {
            "color": color_map.get(at, "#999"), "weight": 2,
            "fillOpacity": 0.15
        },
        tooltip=f"{row['area_type']}"
    ).add_to(m)

# Building footprints
for _, row in bldg_wgs.iterrows():
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        style_function=lambda f: {"color": "#555", "weight": 1, "fillOpacity": 0.4, "fillColor": "#888"}
    ).add_to(m)

# Adjacency edges
for i, j in adjacent_pairs[:200]:  # draw only first 200 to keep map fast
    c1 = bldg_wgs.iloc[i].geometry.centroid
    c2 = bldg_wgs.iloc[j].geometry.centroid
    folium.PolyLine(
        [[c1.y, c1.x], [c2.y, c2.x]],
        color="red", weight=1, opacity=0.5
    ).add_to(m)

# Legend
legend = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border-radius:5px;font-size:12px'>
  <b>Legend</b><br>
  <span style='color:#4472C4'>■</span> Kerrostaloalue (APT)<br>
  <span style='color:#ED7D31'>■</span> Pientaloalue (DTH)<br>
  <span style='color:#888'>■</span> Building footprint<br>
  <span style='color:red'>—</span> Adjacency edge
</div>"""
m.get_root().html.add_child(folium.Element(legend))

m   # displays the interactive map in Jupyter

---
## Part 2 — Building the Knowledge Graph

### 📌 Instruction: KG schema for Finnish buildings

We use the same **Real Estate Core (REC)** and **BOT** ontologies as the Swiss notebook, extended with a custom `fi:` namespace for Finnish-specific concepts:

```
fi:Municipality
     │ rec:hasPart
     ▼
fi:ResidentialArea  (from SYKE: kerrostaloalue / pientaloalue)
     │ rec:hasPart
     ▼
rec:Building         (from OSM)
     │ rec:geometry
     ▼
rec:Polygon
```

The edge we want to **predict** is:
```
rec:Building ──bot:adjacentTo──► rec:Building
```

Note: The original notebook uses `bot:adjacentZone` for rooms. For buildings we use `bot:adjacentTo` — semantically the same concept at a different scale.

In [ ]:
# Define namespaces
REC  = Namespace("https://w3id.org/rec#")
BOT  = Namespace("https://w3id.org/bot#")
FI   = Namespace("https://fi.buildingdata.eu/ontology#")
BASE = Namespace("https://fi.buildingdata.eu/resources/")

# Build the RDF Knowledge Graph
rdf = Graph()
rdf.bind("rec",  REC)
rdf.bind("bot",  BOT)
rdf.bind("fi",   FI)
rdf.bind("base", BASE)

# ── Municipality node ──────────────────────────────────────────────────────────
municipality_uri = URIRef(BASE["municipality/helsinki"])
rdf.add((municipality_uri, RDF.type,   FI["Municipality"]))
rdf.add((municipality_uri, RDFS.label, Literal("Helsinki")))

# ── Residential Area nodes (from SYKE) ────────────────────────────────────────
for _, row in gdf_areas.iterrows():
    area_uri = URIRef(row["uri"])
    rdf.add((area_uri, RDF.type,           FI["ResidentialArea"]))
    rdf.add((area_uri, FI["areaType"],     Literal(row["area_type"])))
    rdf.add((area_uri, FI["areaM2"],       Literal(float(row["area_m2"]))))
    # Link municipality → area
    rdf.add((municipality_uri, REC["hasPart"], area_uri))

# ── Building nodes (from OSM) ──────────────────────────────────────────────────
for _, row in gdf_in_area.iterrows():
    bldg_uri = URIRef(row["uri"])
    rdf.add((bldg_uri, RDF.type,              REC["Building"]))
    rdf.add((bldg_uri, FI["buildingClass"],   Literal(row["building_class"])))
    rdf.add((bldg_uri, FI["areaM2"],          Literal(float(row["area_m2"]))))
    rdf.add((bldg_uri, FI["perimeterM"],      Literal(float(row["perimeter_m"]))))
    rdf.add((bldg_uri, FI["compactness"],     Literal(float(row["compactness"]))))
    rdf.add((bldg_uri, FI["centroidX"],       Literal(float(row["centroid_x"]))))
    rdf.add((bldg_uri, FI["centroidY"],       Literal(float(row["centroid_y"]))))
    # Link area → building
    area_uri = URIRef(row["area_uri"])
    rdf.add((area_uri, REC["hasPart"], bldg_uri))

# ── Adjacency edges (ground truth) ────────────────────────────────────────────
for i, j in adjacent_pairs:
    uri_i = URIRef(gdf_in_area.iloc[i]["uri"])
    uri_j = URIRef(gdf_in_area.iloc[j]["uri"])
    rdf.add((uri_i, BOT["adjacentTo"], uri_j))
    rdf.add((uri_j, BOT["adjacentTo"], uri_i))  # symmetric

print(f"Total RDF triples: {len(rdf):,}")
print(f"  Municipality nodes : 1")
print(f"  Residential areas  : {len(gdf_areas)}")
print(f"  Buildings          : {len(gdf_in_area)}")
print(f"  Adjacency triples  : {len(adjacent_pairs)*2}")

### SPARQL queries on the Finnish KG

### 📌 Instruction: Exploring the KG with SPARQL

Once the KG is built, we can query it with SPARQL regardless of the underlying dataframe. This shows the value of KG representation — the data is now **schema-aware and interoperable**.

In [ ]:
PREFIXES = """
PREFIX rec:  <https://w3id.org/rec#>
PREFIX bot:  <https://w3id.org/bot#>
PREFIX fi:   <https://fi.buildingdata.eu/ontology#>
PREFIX base: <https://fi.buildingdata.eu/resources/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

# Query 1: How many buildings per residential area type?
q1 = rdf.query(PREFIXES + """
    SELECT ?areaType (COUNT(?building) AS ?numBuildings)
    WHERE {
        ?area    fi:areaType  ?areaType .
        ?area    rec:hasPart  ?building .
        ?building a rec:Building .
    }
    GROUP BY ?areaType
    ORDER BY DESC(?numBuildings)
""")
print("Buildings per residential area type:")
for row in q1:
    print(f"  {str(row.areaType):10} → {int(row.numBuildings)} buildings")

# Query 2: Average building area per class
q2 = rdf.query(PREFIXES + """
    SELECT ?class (AVG(?area) AS ?avgArea)
    WHERE {
        ?b a rec:Building ;
           fi:buildingClass ?class ;
           fi:areaM2 ?area .
    }
    GROUP BY ?class
    ORDER BY DESC(?avgArea)
""")
print("\nAverage building area (m²) per building class:")
for row in q2:
    print(f"  {str(row['class']):8} → {float(row.avgArea):.0f} m²")

# Query 3: Buildings with most adjacencies
q3 = rdf.query(PREFIXES + """
    SELECT ?b (COUNT(?neighbor) AS ?numNeighbors)
    WHERE {
        ?b a rec:Building .
        ?b bot:adjacentTo ?neighbor .
    }
    GROUP BY ?b
    ORDER BY DESC(?numNeighbors)
    LIMIT 5
""")
print("\nTop 5 most-connected buildings:")
for row in q3:
    bld_id = str(row.b).split("/")[-1]
    print(f"  Building {bld_id} → {int(row.numNeighbors)} neighbors")

In [ ]:
# Save the KG as a Turtle file for reuse
rdf.serialize("data/finnish_buildings_kg.ttl", format="turtle")
print("KG saved to data/finnish_buildings_kg.ttl")

---
## Part 3 — GNN Dataset Preparation

### 📌 Instruction: Building the PyTorch Geometric HeteroData object

We replicate the same `HeteroData` structure as the Swiss notebook:

| Component | Swiss Dwellings | Finnish Buildings |
|-----------|----------------|------------------|
| Primary node type | `room` | `building` |
| Context node type | `zone`, `polygon` | `residential_area` |
| Edge to predict | `room-adjacentzone-room` | `building-adjacentto-building` |
| Node features | geometry descriptors + BERT | geometry descriptors + one-hot type |

**Node features for buildings (5-dimensional):**
1. `area_m2` — footprint area
2. `perimeter_m` — perimeter length
3. `compactness` — how circular the footprint is
4. `aspect_ratio` — elongation
5. `area_type_enc` — one-hot encoded SYKE area type (APT=0, DTH=1)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Build integer ID lookup tables ─────────────────────────────────────────────
building_uris  = list(gdf_in_area["uri"])
area_uris      = list(gdf_areas["uri"])

bldg_uri2id = {uri: i for i, uri in enumerate(building_uris)}
area_uri2id = {uri: i for i, uri in enumerate(area_uris)}

# ── Building node features ─────────────────────────────────────────────────────
area_type_enc = LabelEncoder().fit_transform(gdf_in_area["area_type"].fillna("UNKNOWN"))

raw_features = np.column_stack([
    gdf_in_area["area_m2"].values,
    gdf_in_area["perimeter_m"].values,
    gdf_in_area["compactness"].values,
    gdf_in_area["aspect_ratio"].values,
    area_type_enc.astype(float),
])

# Standardise (zero mean, unit variance)
scaler = StandardScaler()
building_features = torch.tensor(scaler.fit_transform(raw_features), dtype=torch.float)

# ── Residential area node features ────────────────────────────────────────────
area_type_enc2 = LabelEncoder().fit_transform(gdf_areas["area_type"].fillna("UNKNOWN"))
area_features  = torch.tensor(
    scaler.fit_transform(np.column_stack([
        gdf_areas["area_m2"].values,
        area_type_enc2.astype(float)
    ])), dtype=torch.float
)

print(f"Building feature matrix : {building_features.shape}")
print(f"Area feature matrix     : {area_features.shape}")

In [ ]:
# ── Build edge index tensors ───────────────────────────────────────────────────

# Edge 1: ResidentialArea → hasPart → Building
hasPart_src, hasPart_dst = [], []
for _, row in gdf_in_area.iterrows():
    area_id = area_uri2id.get(row["area_uri"])
    bldg_id = bldg_uri2id[row["uri"]]
    if area_id is not None:
        hasPart_src.append(area_id)
        hasPart_dst.append(bldg_id)

hasPart_edge_index = torch.tensor([hasPart_src, hasPart_dst], dtype=torch.long)

# Edge 2: Building → adjacentTo → Building (all positive pairs)
adj_src = [bldg_uri2id[gdf_in_area.iloc[i]["uri"]] for i, j in adjacent_pairs]
adj_dst = [bldg_uri2id[gdf_in_area.iloc[j]["uri"]] for i, j in adjacent_pairs]

adj_edge_index = torch.tensor([adj_src, adj_dst], dtype=torch.long)

print(f"hasPart edges (area→building): {hasPart_edge_index.shape[1]}")
print(f"adjacentTo edges (building→building): {adj_edge_index.shape[1]}")

In [ ]:
# ── Assemble HeteroData ────────────────────────────────────────────────────────
data = HeteroData()

# Node features
data["building"].x       = building_features
data["building"].node_id = torch.arange(len(building_uris))
data["area"].x           = area_features
data["area"].node_id     = torch.arange(len(area_uris))

# Structural edges (area contains building)
data["area",    "hasPart",    "building"].edge_index = hasPart_edge_index
data["building","partOf",     "area"    ].edge_index = hasPart_edge_index.flip(0)

# Adjacency edges (what we want to predict)
data["building", "adjacentto", "building"].edge_index = adj_edge_index

# Add reverse edges and self-loops for better message passing
data = T.ToUndirected()(data)
data = T.AddSelfLoops()(data)

print(data)
print(f"\nMetadata node types : {data.node_types}")
print(f"Metadata edge types : {data.edge_types}")

### Train / Validation / Test Split

### 📌 Instruction: Stratified split by residential area

Same strategy as the Swiss notebook: split **by area**, not by building. This ensures the model is tested on unseen neighbourhood layouts — not just unseen buildings from layouts it already trained on.

`RandomLinkSplit` from PyTorch Geometric handles the negative sampling automatically — it generates candidate non-edges with label=0.

In [ ]:
# Use PyTorch Geometric's link split transform
# neg_sampling_ratio=1.0: equal number of negative (non-adjacent) pairs
split = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    neg_sampling_ratio=1.0,
    add_negative_train_samples=True,
    edge_types=("building", "adjacentto", "building"),
    rev_edge_types=("building", "rev_adjacentto", "building"),
)

train_data, val_data, test_data = split(data)

print("Dataset split:")
for name, d in [("train", train_data), ("val", val_data), ("test", test_data)]:
    n_edges = d["building", "adjacentto", "building"].edge_label.shape[0]
    n_pos   = d["building", "adjacentto", "building"].edge_label.sum().item()
    print(f"  {name:5}: {n_edges:5} edge candidates  ({int(n_pos)} positive, {n_edges-int(n_pos)} negative)")

In [ ]:
# ── DataLoaders for mini-batch training ───────────────────────────────────────
# LinkNeighborLoader samples subgraphs around target edges (scalable to large graphs)
train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[10, 5],        # sample 10 1-hop neighbors, 5 2-hop
    neg_sampling_ratio=1.0,
    edge_label_index=(
        ("building", "adjacentto", "building"),
        train_data["building", "adjacentto", "building"].edge_label_index
    ),
    edge_label=train_data["building", "adjacentto", "building"].edge_label,
    batch_size=128,
    shuffle=True,
)

val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[10, 5],
    edge_label_index=(
        ("building", "adjacentto", "building"),
        val_data["building", "adjacentto", "building"].edge_label_index
    ),
    edge_label=val_data["building", "adjacentto", "building"].edge_label,
    batch_size=128,
    shuffle=False,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

---
## Part 4 — Graph Neural Network Model

### 📌 Instruction: Same GraphSAGE architecture as the original

We use the **identical** GNN architecture from the Swiss notebook (two-layer GraphSAGE + MLP Classifier). The only change is the node type names (`building`, `area` instead of `room`, `zone`, `polygon`).

This demonstrates the **generalisability** of the architecture — the same model learns to predict room adjacency at interior scale (Swiss) and building adjacency at urban scale (Finnish).

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x


class Classifier(torch.nn.Module):
    def __init__(self, hidden_channels, dropout=0.2):
        super().__init__()
        self.lin1    = torch.nn.Linear(2 * hidden_channels, hidden_channels)
        self.lin2    = torch.nn.Linear(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x_building, edge_label_index):
        # Concatenate source and target building embeddings
        z = torch.cat([x_building[edge_label_index[0]],
                        x_building[edge_label_index[1]]], dim=-1)
        z = self.lin1(z).relu()
        z = self.dropout(z)
        z = self.lin2(z)
        return z.view(-1)


class FinnishBuildingModel(torch.nn.Module):
    def __init__(self, hidden_channels, data):
        super().__init__()
        # Per-node-type input projections
        for node_type in data.node_types:
            in_dim = data[node_type].x.shape[1]
            n_nodes = data[node_type].x.shape[0]
            setattr(self, node_type + "_lin", torch.nn.Linear(in_dim, hidden_channels))
            setattr(self, node_type + "_emb", torch.nn.Embedding(n_nodes, hidden_channels))
        self.metadata   = data.metadata()
        self.gnn        = to_hetero(GNN(hidden_channels), metadata=data.metadata())
        self.classifier = Classifier(hidden_channels)

    def forward(self, data):
        x_dict = {}
        for t in self.metadata[0]:
            lin = getattr(self, t + "_lin")
            emb = getattr(self, t + "_emb")
            x_dict[t] = lin(data[t].x) + emb(data[t].node_id)
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        pred = self.classifier(
            x_dict["building"],
            data["building", "adjacentto", "building"].edge_label_index
        )
        return pred


device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps")  if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {device}")

model = FinnishBuildingModel(hidden_channels=64, data=train_data).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Part 5 — Training

### 📌 Instruction: Training loop

Identical to the Swiss notebook. Binary cross-entropy loss on edge predictions:
- `label=1` → the two buildings ARE adjacent
- `label=0` → the two buildings are NOT adjacent

The model must learn which geometric configurations (size, shape, proximity features encoded in the node embeddings) correspond to adjacency.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 20  # increase to 50-100 for better performance

train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = total_examples = 0

    for batch in train_loader:
        optimizer.zero_grad()
        batch = batch.to(device)
        pred  = model(batch)
        label = batch["building", "adjacentto", "building"].edge_label
        loss  = F.binary_cross_entropy_with_logits(pred, label)
        loss.backward()
        optimizer.step()
        total_loss     += float(loss) * pred.numel()
        total_examples += pred.numel()

    avg_loss = total_loss / total_examples
    train_losses.append(avg_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f}")

print("Training complete.")

---
## Part 6 — Evaluation

### 📌 Instruction: AUC-ROC evaluation

Same metric as the original notebook. AUC > 0.85 is generally considered good. Because we derive adjacency geometrically (not from manual labels), the ground truth here is consistent but not perfect — buildings at exactly the threshold distance may be ambiguously labeled.

In [ ]:
model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in tqdm.tqdm(val_loader, desc="Evaluating"):
        batch = batch.to(device)
        pred  = torch.sigmoid(model(batch))  # convert logit → probability
        label = batch["building", "adjacentto", "building"].edge_label
        preds.append(pred.cpu())
        labels.append(label.cpu())

all_preds  = torch.cat(preds).numpy()
all_labels = torch.cat(labels).numpy()

auc = roc_auc_score(all_labels, all_preds)
print(f"\nValidation AUC-ROC: {auc:.4f}")
print(f"  (0.5 = random, 1.0 = perfect)")

# Save model
with open("data/finnish_model.pkl", "wb") as f:
    pickle.dump(model.cpu(), f)
model = model.to(device)
print("Model saved to data/finnish_model.pkl")

In [ ]:
import matplotlib.pyplot as plt

# Training loss curve
plt.figure(figsize=(8, 3))
plt.plot(train_losses, marker="o", ms=3)
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Training Loss — Finnish Building Adjacency GNN")
plt.tight_layout()
plt.show()

# Prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(all_preds[all_labels == 1], bins=30, alpha=0.7, label="Adjacent (true)",   color="steelblue")
axes[0].hist(all_preds[all_labels == 0], bins=30, alpha=0.7, label="Not adjacent (true)",color="salmon")
axes[0].set_xlabel("Predicted probability")
axes[0].set_title("Prediction distribution")
axes[0].legend()

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(all_labels, all_preds)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1],[0,1],"--",color="grey",label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 7 — Inference: Predict Missing Adjacency

### 📌 Instruction: Simulating the missing-data scenario

We now simulate the original notebook's use case:

> *"What if we have building footprints but no spatial topology?"*

We take one residential area from the test set, remove all its adjacency edges, ask the model to predict them, and compare against ground truth.

In [ ]:
def infer_building_adjacency(
    data: HeteroData,
    building_ids: list,
    threshold: float = 0.5
) -> list:
    """
    Given a list of building node IDs (within one area),
    predict which pairs are adjacent.
    Returns list of (i, j) predicted adjacent pairs.
    """
    data_copy = copy(data)
    # Generate all possible pairs within this area
    candidate_pairs = list(combinations(building_ids, 2))
    if not candidate_pairs:
        return []

    src = torch.tensor([p[0] for p in candidate_pairs], dtype=torch.long)
    dst = torch.tensor([p[1] for p in candidate_pairs], dtype=torch.long)
    data_copy["building", "adjacentto", "building"].edge_label_index = torch.stack([src, dst])
    data_copy = data_copy.to(device)

    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(data_copy)).cpu().numpy()

    return [candidate_pairs[k] for k, s in enumerate(scores) if s >= threshold]


# Pick a residential area from the test set
test_area_idx = 0  # change to try different areas
test_area_uri  = gdf_areas.iloc[test_area_idx]["uri"]
test_area_type = gdf_areas.iloc[test_area_idx]["area_type"]

# Get all buildings in this area
test_buildings = gdf_in_area[gdf_in_area["area_uri"] == test_area_uri]
test_bldg_ids  = [bldg_uri2id[uri] for uri in test_buildings["uri"] if uri in bldg_uri2id]

print(f"Test area     : {test_area_uri.split('/')[-1]} ({test_area_type})")
print(f"Buildings     : {len(test_bldg_ids)}")
print(f"Candidate pairs: {len(list(combinations(test_bldg_ids, 2)))}")

# Predict
predicted_pairs = infer_building_adjacency(test_data, test_bldg_ids, threshold=0.5)
print(f"Predicted adjacent pairs: {len(predicted_pairs)}")

In [ ]:
# ── Visualise predicted vs ground truth for the test area ─────────────────────
id2uri = {v: k for k, v in bldg_uri2id.items()}

# Ground truth: actual adjacent pairs in this area
area_adj_set = set()
for i, j in adjacent_pairs:
    bi = gdf_in_area.iloc[i]["uri"]
    bj = gdf_in_area.iloc[j]["uri"]
    if bi in test_buildings["uri"].values and bj in test_buildings["uri"].values:
        area_adj_set.add((bldg_uri2id[bi], bldg_uri2id[bj]))

predicted_set = {(min(a,b), max(a,b)) for a,b in predicted_pairs}
gt_set        = {(min(a,b), max(a,b)) for a,b in area_adj_set}

tp = predicted_set & gt_set
fp = predicted_set - gt_set
fn = gt_set - predicted_set

precision = len(tp) / max(len(predicted_set), 1)
recall    = len(tp) / max(len(gt_set), 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-8)

print(f"Ground truth adjacencies : {len(gt_set)}")
print(f"Predicted adjacencies    : {len(predicted_set)}")
print(f"True  positives (TP)     : {len(tp)}")
print(f"False positives (FP)     : {len(fp)}")
print(f"False negatives (FN)     : {len(fn)}")
print(f"Precision                : {precision:.3f}")
print(f"Recall                   : {recall:.3f}")
print(f"F1 score                 : {f1:.3f}")

In [ ]:
# ── Map: predicted vs ground truth ─────────────────────────────────────────────
area_bldg_wgs = test_buildings.to_crs(CRS_WGS84)
centre = [area_bldg_wgs.geometry.centroid.y.mean(),
          area_bldg_wgs.geometry.centroid.x.mean()]

m2 = folium.Map(location=centre, zoom_start=16, tiles="CartoDB positron")

# Building footprints
for _, row in area_bldg_wgs.iterrows():
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda f: {"color":"#444","weight":1,"fillColor":"#aaa","fillOpacity":0.5}
    ).add_to(m2)

def draw_edge(m, bldgs_wgs, i, j, color, tooltip):
    try:
        bi = bldgs_wgs[bldgs_wgs["uri"] == id2uri[i]].geometry.centroid.iloc[0]
        bj = bldgs_wgs[bldgs_wgs["uri"] == id2uri[j]].geometry.centroid.iloc[0]
        folium.PolyLine([[bi.y,bi.x],[bj.y,bj.x]], color=color, weight=2.5,
                        opacity=0.8, tooltip=tooltip).add_to(m)
    except (IndexError, KeyError):
        pass

for i, j in tp: draw_edge(m2, area_bldg_wgs, i, j, "green", "TP: correct")
for i, j in fp: draw_edge(m2, area_bldg_wgs, i, j, "orange", "FP: predicted but not true")
for i, j in fn: draw_edge(m2, area_bldg_wgs, i, j, "red",    "FN: missed")

legend2 = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border-radius:5px;font-size:12px'>
  <b>Predicted Adjacency</b><br>
  <span style='color:green'>—</span> True Positive (correct)<br>
  <span style='color:orange'>—</span> False Positive (over-predicted)<br>
  <span style='color:red'>—</span> False Negative (missed)
</div>"""
m2.get_root().html.add_child(folium.Element(legend2))
m2

---
## Part 8 — Graph RAG (Optional)

### 📌 Instruction: Natural language querying of the Finnish building KG

Same pattern as the original notebook — LLM generates SPARQL, runs on the KG, returns a natural language answer.

Set your OpenAI API key via environment variable or `getpass` before running.

In [ ]:
import getpass

# Uncomment to enter key interactively (does not echo to screen)
# OPENAI_API_KEY = getpass.getpass("OpenAI API key: ")

def ask_kg_finland(question: str, rdf_graph, verbose=False) -> str:
    """
    Ask a natural language question about the Finnish building KG.
    The LLM generates SPARQL, which is executed on the KG.
    """
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    schema_description = """
    The knowledge graph contains:
    - fi:Municipality nodes (label: city name)
    - fi:ResidentialArea nodes (fi:areaType: 'APT' or 'DTH', fi:areaM2: area in m²)
    - rec:Building nodes (fi:buildingClass: 'APT'|'COMM'|'OTHER', fi:areaM2, fi:compactness, fi:perimeterM)
    - Edges: rec:hasPart (municipality→area, area→building), bot:adjacentTo (building→building)

    Prefixes:
      PREFIX rec:  <https://w3id.org/rec#>
      PREFIX bot:  <https://w3id.org/bot#>
      PREFIX fi:   <https://fi.buildingdata.eu/ontology#>
      PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    """

    prompt = f"""
You are a SPARQL expert working with a Finnish building Knowledge Graph.

Schema:
{schema_description}

Generate a valid SPARQL SELECT query to answer:
{question}

Return ONLY the SPARQL query, no explanation.
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    sparql_query = response.choices[0].message.content.strip()
    sparql_query = sparql_query.replace("```sparql","").replace("```","").strip()

    if verbose:
        print("Generated SPARQL:")
        print(sparql_query)

    try:
        results = list(rdf_graph.query(sparql_query))
        result_str = str(results[:10])  # limit output
    except Exception as e:
        result_str = f"Query error: {e}"

    answer_prompt = f"Question: {question}\nSPARQL result: {result_str}\nAnswer in one sentence:"
    answer = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": answer_prompt}]
    )
    return answer.choices[0].message.content.strip()


# Example questions for the Finnish building KG:
example_questions = [
    "How many buildings are in the dataset?",
    "What is the average building area for APT class buildings?",
    "How many residential areas of type 'APT' are there?",
    "Which buildings have more than 5 adjacent neighbours?",
]

print("Example questions you can ask:")
for q in example_questions:
    print(f"  → {q}")

# Uncomment to run:
# answer = ask_kg_finland(example_questions[0], rdf, verbose=True)
# print("\nAnswer:", answer)

---
## Appendix A — Extension to Room-Level with IFC Files

### 📌 Instruction: How to replicate the original Swiss notebook at room level

To exactly replicate the room-adjacency task for Finnish buildings, you need **IFC files** with `IfcSpace` elements.

#### Where to get Finnish IFC files

| Source | Notes |
|--------|-------|
| **ARA (Housing Finance and Development Centre)** | Publicly funded housing projects often have open BIM |
| **Helsinki City open BIM** | Some city buildings available at hri.fi |
| **University BIM projects** | Aalto, Tampere University research datasets |
| **buildingSMART Finland** | Sample IFC files for testing |
| **Your own project collaborators** | Most realistic for thesis work |

#### Install `ifcopenshell`

```bash
pip install ifcopenshell
```

Or use the conda package (recommended for full geometry support):
```bash
conda install -c conda-forge ifcopenshell
```

In [ ]:
# ── IFC → Finnish Room KG (template — requires ifcopenshell and an IFC file) ───

def parse_ifc_to_room_df(ifc_path: str) -> pd.DataFrame:
    """
    Parse an IFC file and extract room-level data as a DataFrame.
    Returns one row per IfcSpace with geometry and hierarchy info.
    """
    import ifcopenshell
    import ifcopenshell.util.element as util
    import ifcopenshell.geom

    ifc  = ifcopenshell.open(ifc_path)
    rows = []

    for space in ifc.by_type("IfcSpace"):
        storey   = util.get_container(space)        # IfcBuildingStorey
        building = util.get_container(storey) if storey else None
        site     = util.get_container(building) if building else None

        rows.append({
            "space_id":       space.GlobalId,
            "space_name":     space.Name or "",        # e.g. "Keittiö"
            "space_longname": space.LongName or "",
            "storey_id":      storey.GlobalId if storey else None,
            "building_id":    building.GlobalId if building else None,
            "site_id":        site.GlobalId if site else None,
        })

    return pd.DataFrame(rows)


def get_ifc_adjacencies(ifc_path: str) -> list[tuple]:
    """
    Extract room-room adjacencies from IfcRelSpaceBoundary.
    Returns list of (space_id_1, space_id_2) tuples.
    """
    import ifcopenshell
    ifc = ifcopenshell.open(ifc_path)
    adjacencies = set()

    # IfcRelSpaceBoundary links a space to its bounding element (wall/slab)
    # Two spaces sharing the same boundary element are adjacent
    element_to_spaces = {}
    for rel in ifc.by_type("IfcRelSpaceBoundary"):
        space   = rel.RelatingSpace
        element = rel.RelatedBuildingElement
        if space and element:
            element_id = element.GlobalId
            space_id   = space.GlobalId
            if element_id not in element_to_spaces:
                element_to_spaces[element_id] = []
            element_to_spaces[element_id].append(space_id)

    for element_id, spaces in element_to_spaces.items():
        for s1, s2 in combinations(spaces, 2):
            adjacencies.add((min(s1, s2), max(s1, s2)))

    return list(adjacencies)


# Usage example:
# df_rooms = parse_ifc_to_room_df("path/to/building.ifc")
# adjacencies = get_ifc_adjacencies("path/to/building.ifc")
#
# Then follow the same KG + GNN pipeline as above,
# replacing gdf_buildings with df_rooms and
# adjacent_pairs with adjacencies.

print("IFC parsing functions defined.")
print("Provide an IFC file path and call parse_ifc_to_room_df() + get_ifc_adjacencies()")

---
## Appendix B — NLS Finland Topographic Database (API key required)

### 📌 Instruction: Using official Finnish building data instead of OSM

The **NLS Finland OGC API** gives access to official building footprints from the National Topographic Database. Register for a free API key at: https://www.maanmittauslaitos.fi/en/rajapinnat/api-avaimen-ohje

Buildings are available as the `rakennus` (building) feature type.

In [ ]:
# NLS Finland OGC API Features — requires free API key
# Register at: https://www.maanmittauslaitos.fi/en/rajapinnat/api-avaimen-ohje

NLS_API_BASE = "https://avoin-paikkatieto.maanmittauslaitos.fi/maastotiedot/features/v1"

def fetch_nls_buildings(api_key: str, bbox: tuple, limit: int = 1000) -> gpd.GeoDataFrame:
    """
    Fetch building footprints from NLS Finland Topographic Database.
    bbox: (min_lon, min_lat, max_lon, max_lat) in WGS84
    Requires a free API key from maanmittauslaitos.fi
    """
    url = f"{NLS_API_BASE}/collections/rakennus/items"
    params = {
        "bbox":   f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
        "limit":  limit,
        "f":      "json",
    }
    headers = {"Authorization": f"Bearer {api_key}"}
    resp = requests.get(url, params=params, headers=headers, timeout=60)
    resp.raise_for_status()
    gdf  = gpd.GeoDataFrame.from_features(resp.json()["features"], crs=CRS_WGS84)
    return gdf.to_crs(CRS_METRIC)


# Helsinki bounding box in WGS84
HELSINKI_BBOX_WGS84 = (24.83, 60.10, 25.25, 60.32)

# Usage:
# NLS_API_KEY = getpass.getpass("NLS API key: ")
# gdf_nls_buildings = fetch_nls_buildings(NLS_API_KEY, HELSINKI_BBOX_WGS84)
# print(gdf_nls_buildings.columns.tolist())  # explore available attributes

print("NLS fetch function defined.")
print("Provide your NLS API key to use official Finnish building data.")

---
## Summary

### What this notebook demonstrated

| Step | Done | Key tool |
|------|------|----------|
| Download Finnish residential areas (SYKE WFS) | ✓ | `requests` + `geopandas` |
| Download building footprints (OSM) | ✓ | `osmnx` |
| Compute spatial adjacency (buffer intersection) | ✓ | `shapely` |
| Build Knowledge Graph (REC + BOT ontologies) | ✓ | `rdflib` |
| Query KG with SPARQL | ✓ | `rdflib` |
| Train heterogeneous GNN (GraphSAGE) | ✓ | `torch-geometric` |
| Evaluate with AUC-ROC | ✓ | `sklearn` |
| Infer missing adjacency edges | ✓ | trained model |
| Graph RAG | template | `openai` |
| Room-level extension | template | `ifcopenshell` |

### Limitations and next steps

1. **Scale:** SYKE operates at 1:100,000 — many small buildings may not align with area polygons
2. **Adjacency definition:** Buffer-based adjacency is a proxy. IFC `IfcRelSpaceBoundary` is more semantically precise
3. **Node features:** Only geometric features are used. Add: number of floors (from NLS), year of construction, energy class
4. **Task extension:** Add node classification (predict building type from geometry only), or graph-level regression (predict energy performance per neighbourhood)
5. **Room scale:** Obtain IFC files and use `parse_ifc_to_room_df()` from Appendix A for the original room-level task

### Key data sources for Finnish building research

| Resource | URL |
|----------|-----|
| SYKE residential areas | https://ckan.ymparisto.fi/dataset/asuinalueet |
| NLS 3D Buildings (LoD2) | https://www.maanmittauslaitos.fi/en/maps-and-spatial-data/datasets-and-interfaces/product-descriptions/buildings-3d |
| NLS Topographic Database | https://avoin-paikkatieto.maanmittauslaitos.fi/maastotiedot/features/v1 |
| Helsinki open data | https://hri.fi |
| Building topology ontology | https://w3id.org/bot |
| Real Estate Core ontology | https://www.realestatecore.io |